In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Lab3_Cleaning").getOrCreate()

df = spark.read.format("delta").load("abfss://lakehouse@goodreadreviews60312547.dfs.core.windows.net/gold/curated_reviews")


In [0]:
from pyspark.sql.functions import col, trim, length, when, lower, regexp_replace

# 1️⃣ Remove null or blank rows
df_clean = df.dropna(subset=["rating", "review_text", "book_id", "author_id"])

# 2️⃣ Remove duplicates
df_clean = df_clean.dropDuplicates(["review_id"])

# 3️⃣ Normalize text fields (trim, lowercase, clean spaces)
text_cols = ["title", "name", "review_text"]
for c in text_cols:
    df_clean = df_clean.withColumn(c, trim(col(c)))
    df_clean = df_clean.withColumn(c, regexp_replace(col(c), "\s+", " "))

# Optional lowercase normalization
df_clean = df_clean.withColumn("language", lower(col("language")))

# 4️⃣ Replace missing or empty languages
df_clean = df_clean.withColumn("language", when(col("language").isNull(), "unknown")
                               .when(col("language") == "", "unknown")
                               .otherwise(col("language")))

# 5️⃣ Remove short or meaningless reviews (<10 characters)
df_clean = df_clean.filter(length(col("review_text")) >= 10)

# 6️⃣ Cast and fix data types
df_clean = df_clean.withColumn("rating", col("rating").cast("float"))
df_clean = df_clean.withColumn("book_id", col("book_id").cast("string"))
df_clean = df_clean.withColumn("author_id", col("author_id").cast("string"))
df_clean = df_clean.withColumn("user_id", col("user_id").cast("string"))


<>:13: SyntaxWarning: invalid escape sequence '\s'
<>:13: SyntaxWarning: invalid escape sequence '\s'
/home/spark-4f71d0e3-1bf4-4457-970d-a1/.ipykernel/2599/command-5778612640754848-4134662719:13: SyntaxWarning: invalid escape sequence '\s'
  df_clean = df_clean.withColumn(c, regexp_replace(col(c), "\s+", " "))
<unknown>:13: SyntaxWarning: invalid escape sequence '\s'


In [0]:
df = spark.read.format("delta").load(
  "abfss://lakehouse@goodreadreviews60312547.dfs.core.windows.net/gold/curated_reviews"
)


In [0]:
from pyspark.sql.functions import avg, count, split, size

spark.conf.set( 
"fs.azure.account.key.goodreadreviews60312547.dfs.core.windows.net", 
"sKB0IP98VHBh/o5Lg2plrnJzvytb9/liAATVm6l4Lqhsqu0kCBBz/HVTVF9wB8mBMMsKprQhK2K/+AStFrdTYQ==" 
) 

df = spark.read.format("delta").load(
  "abfss://lakehouse@goodreadreviews60312547.dfs.core.windows.net/gold/curated_reviews"
)

df_clean = df.persist()

# Review length in characters
df_clean = df_clean.withColumn("review_length", length(col("review_text")))

# Word count
df_clean = df_clean.withColumn("word_count", size(split(col("review_text"), " ")))

# Aggregation: average rating and number of reviews per book_id
book_features = df_clean.groupBy("book_id").agg(
    avg("rating").alias("avg_rating_book"),
    count("review_id").alias("num_reviews_book")
)

# Join aggregated features back to main dataframe
df_enriched = df_clean.join(book_features, on="book_id", how="left")


In [0]:
output_path = "abfss://lakehouse@goodreadreviews60312547.dfs.core.windows.net/gold/features_v1"

df_enriched.write.format("delta").mode("overwrite").save(output_path)


In [0]:
df_check = spark.read.format("delta").load(output_path)

df_check.printSchema()
df_check.show(5)


root
 |-- book_id: string (nullable = true)
 |-- review_id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- author_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- rating: integer (nullable = true)
 |-- review_text: string (nullable = true)
 |-- language: string (nullable = true)
 |-- book_avg_rating: string (nullable = true)
 |-- book_ratings_count: string (nullable = true)
 |-- review_length: integer (nullable = true)
 |-- word_count: integer (nullable = true)
 |-- avg_rating_book: double (nullable = true)
 |-- num_reviews_book: long (nullable = true)

+--------+--------------------+--------------------+-----------+----+--------------------+------+--------------------+--------+---------------+------------------+-------------+----------+-----------------+----------------+
| book_id|           review_id|               title|  author_id|name|             user_id|rating|         review_text|language|book_avg_rat